In [ ]:
import torch
import os
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset
from torchvision import transforms
import torch.optim as optim
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import random
import pickle
from sklearn.metrics import matthews_corrcoef, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset
import re
from collections import defaultdict
from tqdm import tqdm
from torchinfo import summary
from frz_predictor import generate_frz_training_dataset, FreezeDataset, initialize_mamba2_predictor, initialize_smartfrz_predictor
        
def main(args):
  total_count = args.number_of_samples
  window_size = args.context_window_size
  combined_training_dataset = []
  for experiment_name, should_generate_training_data in zip(args.experiment_names, args.generate_training_data):
      root_dir = f"{experiment_name}/context_window_{args.context_window_size}"
        
      if should_generate_training_data:
        generate_frz_training_dataset(root_dir, total_count, args.frz_predictor_type)
      train_dataset = FreezeDataset(f"{root_dir}/dataset_{args.frz_predictor_type}.pkl", args.frz_predictor_type)
      combined_training_dataset.append(train_dataset)
  train_dataset = ConcatDataset(combined_training_dataset)
        
  all_indices = list(range(len(train_dataset)))
    
  seed_to_indices = defaultdict(list)
  for idx in range(len(train_dataset)):
    _, _, _, seed = train_dataset[idx][0]
    seed_to_indices[seed].append(idx)
  seed_list = list(seed_to_indices.keys())
  print("The seeds in the dataset: ", seed_list, len(seed_list))
  print("The number of entries per seed: ", [len(seed_indices) for seed_indices in seed_to_indices.values()])

  # REPRODUCIBILITY WITH RANDOM SEED
  random.seed(1234)
  #### NEW CODE
  number_of_seeds_for_testing_dataset = int(args.percentage_of_seeds_for_validation * len(seed_list))
  chosen_seeds = set(random.sample(seed_list, number_of_seeds_for_testing_dataset))
  training_dataset_seeds = set(seed_list) - chosen_seeds

  train_indices = []
  val_indices = []

  for seed in chosen_seeds:
    val_indices.extend(seed_to_indices[seed])
  for seed in training_dataset_seeds:
    train_indices.extend(seed_to_indices[seed])

  train_subset = Subset(train_dataset, train_indices)
  val_subset = Subset(train_dataset, val_indices)
    
  device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    
  batch_size = 32
  num_workers = 0
  train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
  val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
  print(f"Number of batches in Train Loader: {len(train_loader)}")
  
  if args.frz_predictor_type == "smartfrz":
    re_size = args.re_size
    in_channel = re_size
    hid_channel = 256
    out_channel = 64
    predictor = initialize_smartfrz_predictor(in_channel, hid_channel, out_channel)
    if args.use_pretrained_model:
        predictor.load_state_dict(torch.load(args.pretrained_weights, map_location=device))
  elif args.frz_predictor_type == "mambafrz":
    feature_dim = args.re_size
    mlp_hid_channel = 512
    mlp_out_channel = 2
    ssm_state_expansion_factor = 32
    projected_dim = feature_dim // 2
    predictor = initialize_mamba2_predictor(feature_dim=feature_dim, projected_dim=projected_dim, ssm_state_expansion_factor=ssm_state_expansion_factor, mlp_hid_channel=mlp_hid_channel, mlp_out_channel=mlp_out_channel)
    if args.use_pretrained_model:
        predictor.load_state_dict(torch.load(args.pretrained_weights, map_location=device))
  # Simple parameter count
  total_params = sum(p.numel() for p in predictor.parameters())
  trainable_params = sum(p.numel() for p in predictor.parameters() if p.requires_grad)
  # print(summary(predictor, input_size=(1, args.context_window_size, args.re_size)))

  print(f"Total parameters: {total_params:,}")
  print(f"Trainable parameters: {trainable_params:,}")
  
  predictor.to(device)

  label_smoothing = 0.2 # included label smoothing

  criterion = nn.CrossEntropyLoss()
  criterion = criterion.to(device)

  if args.frz_predictor_type == "smartfrz":
    if args.use_pretrained_model:
        lr = 1e-4
    else:
        lr = 1e-3

    optimizer = optim.AdamW(predictor.parameters(),
                        lr=lr,
                        weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, verbose=True)
  elif args.frz_predictor_type == "mambafrz":
    if args.use_pretrained_model:
        lr = 1e-5
    else:
        lr = 1e-3
    
    optimizer = optim.AdamW(predictor.parameters(),
                        lr=lr,
                        weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, verbose=True)
    
  num_epochs = args.num_epochs
  
  model_save_path = f"{args.folder_to_save_checkpoints}/{args.frz_predictor_type}"
  os.makedirs(model_save_path, exist_ok=True)
  
  frozen_count = 0
  non_frozen_count = 0
  best_validation_accuracy = 0.0
  
  training_epoch_loss = []
  for epoch in range(num_epochs):
    predictor.train()
    train_running_loss = 0.0
    correct = 0
    total = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)
    for inputs, labels in progress_bar:
      layer_names_list = inputs[1]
      epoch_list = inputs[2]
      seed_list = inputs[3]
      inputs, labels = inputs[0].to(device), labels.to(device)
      optimizer.zero_grad()
      outputs = predictor(inputs)
      loss = criterion(outputs, labels)
      loss.backward()
      torch.nn.utils.clip_grad_norm_(predictor.parameters(), max_norm=1.0)
      optimizer.step()
      train_running_loss += loss.item()
      correct += sum([torch.argmax(pred).item() == label.item() for pred, label in zip(outputs, labels)])
      total += labels.size(0)
    train_running_loss /= len(train_loader)
    training_epoch_loss.append(train_running_loss)
    epoch_accuracy = correct / total
    print(f"Epoch {epoch + 1}/{num_epochs}, Training Loss: {train_running_loss:.4f}, Training Accuracy: {epoch_accuracy:.4f}")
    
    # Begin validation
    predictor.eval()
    seed_prediction_tracker = {} # Each key is a seed to another dict, which stores layers and predictions there
    layer_by_layer_accuracy = {} # Get accuracy per layer name, that way it shows overall trends across multiple seeds
    
    progress_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)
    val_running_loss = 0.0
    val_total_correct = 0
    val_total_num = 0
    
    with torch.no_grad():
        for inputs, labels in progress_bar:
            seed_list = inputs[3]
            layer_names_list = inputs[1]
            epoch_numbers_list = inputs[2]
            inputs, labels = inputs[0].to(device), labels.to(device)
            outputs = predictor(inputs)
            loss = criterion(outputs, labels)
            val_running_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            
            for seed, pred, label, layer_name, epoch_num in zip(seed_list, preds, labels, layer_names_list, epoch_numbers_list):
                if seed not in seed_prediction_tracker:
                    seed_prediction_tracker[seed] = {}
                if layer_name not in seed_prediction_tracker[seed]:
                    seed_prediction_tracker[seed][layer_name] = []
                # Set layer by layer accuracy defaults
                if layer_name not in layer_by_layer_accuracy:
                    layer_by_layer_accuracy[layer_name] = {"total_correct": 0, "total_samples": 0}
                if pred.item() == label.item():
                    val_total_correct += 1
                    layer_by_layer_accuracy[layer_name]["total_correct"] += 1
                seed_prediction_tracker[seed][layer_name].append((epoch_num, pred.item(), label.item()))
                val_total_num += 1
                layer_by_layer_accuracy[layer_name]["total_samples"] += 1
    # Counter for number of seeds
    counter = args.number_of_seeds_to_plot
    for seed in seed_prediction_tracker.keys():
        if counter == 0:
            break
        for layer_name, frz_predictions_by_layer in seed_prediction_tracker[seed].items():
            frz_predictions_by_layer.sort(key=lambda item: int(item[0]))
            epoch_list = [int(item[0]) for item in frz_predictions_by_layer]
            frz_predictor_list = [item[1] for item in frz_predictions_by_layer]
            label_predictor_list = [item[2] for item in frz_predictions_by_layer]
            
            plt.title(f"Seed {seed}, Layer {layer_name} Predictions")
            name_of_predictor = "MambaFRZ" if args.frz_predictor_type == "mambafrz" else "SmartFRZ"
            plt.plot(epoch_list, frz_predictor_list, label=f"{name_of_predictor} Predictions")
            plt.plot(epoch_list, label_predictor_list, label="Labels")
            plt.legend()
            plt.show()
        counter -= 1
    val_running_loss /= len(val_loader)
    scheduler.step(val_running_loss)
    
    print(f"Validation Accuracy by Layer for Epoch {epoch + 1}:")

    for layer_name, layer_accuracy in layer_by_layer_accuracy.items():
        acc = layer_accuracy["total_correct"] / layer_accuracy["total_samples"]
        print(f"{layer_name}: {acc:.4f}, {layer_accuracy['total_correct']} / {layer_accuracy['total_samples']}")
    
    validation_accuracy = val_total_correct / val_total_num
    print(f"Overall Validation Accuracy: {(validation_accuracy):.4f}")
    
    if validation_accuracy > best_validation_accuracy:
      best_validation_accuracy = epoch_accuracy
      torch.save(predictor.state_dict(), os.path.join(model_save_path, f"{args.frz_predictor_type}_{epoch}.pth"))
      print(f"New Best Acc: {best_validation_accuracy}")
  plt.plot(training_epoch_loss, label='Training Loss')
  plt.legend()
  plt.show()
      
class Args:
  def __init__(self, 
               experiment_names, 
               context_window_size, 
               number_of_samples, 
               re_size=1024, 
               num_epochs=2, 
               generate_training_data=False, 
               checkpoint_folder="checkpoints", 
               frz_predictor_type="smartfrz",
               use_pretrained_model=False,
               pretrained_weights="text.pth",
               percentage_of_seeds_for_validation=0.2,
               number_of_seeds_to_plot=2,
               folder_to_save_checkpoints="checkpoints_folder"
            ):
    self.context_window_size = context_window_size
    self.experiment_names = experiment_names
    self.number_of_samples = number_of_samples
    self.re_size = re_size
    self.num_epochs = num_epochs
    self.generate_training_data = generate_training_data
    self.checkpoint_folder = checkpoint_folder
    self.frz_predictor_type = frz_predictor_type
    self.use_pretrained_model = use_pretrained_model
    self.pretrained_weights = pretrained_weights
    self.percentage_of_seeds_for_validation = percentage_of_seeds_for_validation
    self.number_of_seeds_to_plot = number_of_seeds_to_plot
    self.folder_to_save_checkpoints = folder_to_save_checkpoints

args = Args(
    experiment_names=["cifar100_frz_predictor_training_dataset/vgg16/data_generation/training_data", "cifar100_frz_predictor_training_dataset/vgg11/data_generation/training_data_more_data_times_three"], 
    context_window_size=30, 
    number_of_samples=50000,
    re_size=1024, 
    num_epochs=10, 
    generate_training_data=[True, True], 
    checkpoint_folder="from_finetuning_do_not_interfere", 
    frz_predictor_type="mambafrz", 
    use_pretrained_model=False,
    pretrained_weights="mambafrz_vgg11_data_generation_12_seeds/training_data_more_data/context_window_30/mambafrz_small_test/mambafrz_9.pth",
    percentage_of_seeds_for_validation=0.2,
    number_of_seeds_to_plot=2,
    folder_to_save_checkpoints="training_frz_predictors/combined_dataset_vgg11_and_vgg16"
)
main(args)